# 02 — EfficientNetB0 avec CLAHE conditionnel

Ce notebook répète exactement le protocole de la baseline en modifiant **une seule variable** : les images
dont l'écart-type du canal de luminance LAB est inférieur à 40 reçoivent un CLAHE. Cette expérience d'ablation
dira si le traitement améliore réellement la généralisation.

OpenCV est appelé avec `tf.numpy_function` : cette variante peut être plus lente côté CPU. Il faut comparer
les métriques, mais aussi le temps par époque.


In [1]:
from pathlib import Path
import json
import os
import random

import certifi
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Python installé depuis python.org sur macOS ne référence pas toujours le trousseau
# système. On fournit à urllib/Keras un bundle CA explicite sans désactiver SSL.
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

def find_project_root() -> Path:
    """Permet d'exécuter le notebook depuis sa racine ou son propre dossier."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "raw_x.csv").exists() and (candidate / "images").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable (data/raw_x.csv et images/ attendus).")

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
IMAGE_DIR = ROOT / "images" / "image_train"
ARTIFACTS_DIR = ROOT / "artifacts" / "image_classification"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("KERAS_HOME", str(ARTIFACTS_DIR / ".keras"))

print("Racine :", ROOT)
print("Artefacts :", ARTIFACTS_DIR)
print("Certificats SSL :", os.environ["SSL_CERT_FILE"])


Racine : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2
Artefacts : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification
Certificats SSL : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/.conda/rakuten-metal/lib/python3.10/site-packages/certifi/cacert.pem


In [2]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)

TARGET_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

with open(ARTIFACTS_DIR / "class_mapping.json", encoding="utf-8") as handle:
    class_mapping = json.load(handle)

index_to_code = [int(value) for value in class_mapping["index_to_code"]]
code_to_index = {code: index for index, code in enumerate(index_to_code)}
NUM_CLASSES = len(index_to_code)

manifest = pd.read_csv(ARTIFACTS_DIR / "image_manifest.csv")
manifest["class_index"] = manifest["prdtypecode"].map(code_to_index)
assert manifest["class_index"].notna().all()

def decode_resize(path, label, weight):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize_with_pad(image, *TARGET_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)  # EfficientNet attend des pixels dans [0, 255].
    return image, tf.cast(label, tf.int32), tf.cast(weight, tf.float32)

def make_dataset(frame, training=False, transform=None, weighted=False):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["class_index"].astype("int32").to_numpy()
    weights = frame["sample_weight"].astype("float32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if transform is not None:
        ds = ds.map(transform, num_parallel_calls=AUTOTUNE)
    if not weighted:
        ds = ds.map(lambda x, y, w: (x, y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_df = manifest.query("split == 'train'").copy()
val_df = manifest.query("split == 'validation'").copy()
print(f"Train={len(train_df):,} | Validation={len(val_df):,} | Classes={NUM_CLASSES}")


Train=65,900 | Validation=16,475 | Classes=27


## Prétraitement CLAHE


In [3]:
import cv2

CONTRAST_THRESHOLD = 40.0

def clahe_numpy(image):
    image_uint8 = np.clip(image, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(image_uint8, cv2.COLOR_RGB2LAB)
    lightness, channel_a, channel_b = cv2.split(lab)
    if float(lightness.std()) >= CONTRAST_THRESHOLD:
        return image_uint8.astype(np.float32)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(lightness)
    result = cv2.cvtColor(cv2.merge((enhanced, channel_a, channel_b)), cv2.COLOR_LAB2RGB)
    return result.astype(np.float32)

def apply_clahe_tf(image, label, weight):
    image = tf.numpy_function(clahe_numpy, [image], Tout=tf.float32)
    image.set_shape((*TARGET_SIZE, 3))
    return image, label, weight


In [4]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model():
    augmentation = keras.Sequential(
        [
            layers.RandomRotation(0.03, fill_mode="reflect", seed=SEED),
            layers.RandomZoom(0.10, fill_mode="reflect", seed=SEED),
            layers.RandomContrast(0.10, seed=SEED),
        ],
        name="augmentation",
    )
    inputs = keras.Input(shape=(*TARGET_SIZE, 3), name="image")
    x = augmentation(inputs)
    local_weights = ARTIFACTS_DIR / ".keras" / "models" / "efficientnetb0_notop.h5"
    weights = str(local_weights) if local_weights.exists() else "imagenet"
    print("Poids EfficientNet :", weights)
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights=weights,
        input_shape=(*TARGET_SIZE, 3),
    )
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(0.30, seed=SEED)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="prdtypecode")(x)
    return keras.Model(inputs, outputs), backbone

def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

def callbacks_for(run_dir):
    run_dir.mkdir(parents=True, exist_ok=True)
    return [
        keras.callbacks.ModelCheckpoint(
            run_dir / "best_model.keras", monitor="val_loss", save_best_only=True
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7
        ),
        keras.callbacks.CSVLogger(run_dir / "training_log.csv", append=True),
    ]

def unfreeze_last_layers(backbone, count=30):
    backbone.trainable = True
    for layer in backbone.layers[:-count]:
        layer.trainable = False
    # Les statistiques de BatchNorm restent figées sur ce dataset relativement petit.
    for layer in backbone.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False


In [5]:
RUN_NAME = "efficientnet_clahe"
TRAIN_TRANSFORM = apply_clahe_tf
VAL_TRANSFORM = apply_clahe_tf


In [6]:
RUN_DIR = ARTIFACTS_DIR / RUN_NAME
train_ds = make_dataset(train_df, training=True, transform=TRAIN_TRANSFORM, weighted=True)
val_ds = make_dataset(val_df, training=False, transform=VAL_TRANSFORM, weighted=False)
run_callbacks = callbacks_for(RUN_DIR)

model, backbone = build_model()
compile_model(model, learning_rate=1e-3)
model.summary()


2026-09-11 11:16:34.296911: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-09-11 11:16:34.297096: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-11 11:16:34.297101: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-09-11 11:16:34.297529: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-11 11:16:34.297545: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Poids EfficientNet : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification/.keras/models/efficientnetb0_notop.h5


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prdtypecode (Dense)             │ (None, 27)             │        34,587 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,084,158 (15.58 MB)

 Trainable params: 34,587 (135.11 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## Phase 1 — Backbone gelé


In [7]:
EPOCHS_FROZEN = 8
history_frozen = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FROZEN,
    callbacks=run_callbacks,
)


Epoch 1/8


2026-09-11 11:16:38.458806: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


2060/2060 ━━━━━━━━━━━━━━━━━━━━ 423s 202ms/step - accuracy: 0.4791 - loss: 1.8093 - top3_accuracy: 0.7086 - val_accuracy: 0.5433 - val_loss: 1.5355 - val_top3_accuracy: 0.7698 - learning_rate: 0.0010
Epoch 2/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 385s 186ms/step - accuracy: 0.5313 - loss: 1.6069 - top3_accuracy: 0.7588 - val_accuracy: 0.5619 - val_loss: 1.4848 - val_top3_accuracy: 0.7789 - learning_rate: 0.0010
Epoch 3/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 414s 201ms/step - accuracy: 0.5432 - loss: 1.5698 - top3_accuracy: 0.7664 - val_accuracy: 0.5623 - val_loss: 1.4767 - val_top3_accuracy: 0.7826 - learning_rate: 0.0010
Epoch 4/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 401s 194ms/step - accuracy: 0.5451 - loss: 1.5517 - top3_accuracy: 0.7714 - val_accuracy: 0.5608 - val_loss: 1.4728 - val_top3_accuracy: 0.7811 - learning_rate: 0.0010
Epoch 5/8
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 344s 167ms/step - accuracy: 0.5471 - loss: 1.5438 - top3_accuracy: 0.7736 - val_accuracy: 0.5728 - val_loss: 1.4468 - val_top3_accura

## Phase 2 — Fine-tuning partiel


In [8]:
unfreeze_last_layers(backbone, count=30)
compile_model(model, learning_rate=1e-5)

start_epoch = len(history_frozen.history["loss"])
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=start_epoch,
    epochs=start_epoch + 12,
    callbacks=run_callbacks,
)

model.save(RUN_DIR / "final_model.keras")
print("Modèle sauvegardé dans", RUN_DIR)


Epoch 9/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 403s 191ms/step - accuracy: 0.5729 - loss: 1.4420 - top3_accuracy: 0.7947 - val_accuracy: 0.5843 - val_loss: 1.3906 - val_top3_accuracy: 0.8016 - learning_rate: 1.0000e-05
Epoch 10/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 390s 189ms/step - accuracy: 0.5828 - loss: 1.4034 - top3_accuracy: 0.8016 - val_accuracy: 0.5860 - val_loss: 1.3774 - val_top3_accuracy: 0.8038 - learning_rate: 1.0000e-05
Epoch 11/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 385s 187ms/step - accuracy: 0.5932 - loss: 1.3677 - top3_accuracy: 0.8090 - val_accuracy: 0.5915 - val_loss: 1.3599 - val_top3_accuracy: 0.8069 - learning_rate: 1.0000e-05
Epoch 12/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 400s 194ms/step - accuracy: 0.5997 - loss: 1.3459 - top3_accuracy: 0.8129 - val_accuracy: 0.5962 - val_loss: 1.3449 - val_top3_accuracy: 0.8101 - learning_rate: 1.0000e-05
Epoch 13/20
2060/2060 ━━━━━━━━━━━━━━━━━━━━ 449s 218ms/step - accuracy: 0.6053 - loss: 1.3255 - top3_accuracy: 0.8179 - val_accuracy: 0.5992 -

## Contrôle rapide


In [9]:
from sklearn.metrics import classification_report, f1_score

best_model = keras.models.load_model(RUN_DIR / "best_model.keras")
probabilities = best_model.predict(val_ds, verbose=1)
y_pred = probabilities.argmax(axis=1)
y_true = val_df["class_index"].to_numpy()

print("F1 weighted :", f1_score(y_true, y_pred, average="weighted"))
print("F1 macro    :", f1_score(y_true, y_pred, average="macro"))
print(classification_report(y_true, y_pred, target_names=[str(x) for x in index_to_code]))


515/515 ━━━━━━━━━━━━━━━━━━━━ 60s 108ms/step
F1 weighted : 0.6032315927152736
F1 macro    : 0.5603441980914924
              precision    recall  f1-score   support

          10       0.46      0.65      0.53       591
          40       0.53      0.53      0.53       476
          50       0.48      0.38      0.42       320
          60       0.68      0.66      0.67       164
        1140       0.65      0.64      0.64       510
        1160       0.89      0.89      0.89       768
        1180       0.62      0.30      0.41       148
        1280       0.45      0.42      0.43       897
        1281       0.44      0.21      0.29       396
        1300       0.67      0.70      0.69       940
        1301       0.64      0.53      0.58       150
        1302       0.45      0.34      0.39       488
        1320       0.46      0.46      0.46       636
        1560       0.56      0.59      0.57       996
        1920       0.78      0.81      0.79       853
        1940       0.62  